# 04 · Concept Pooling and Compression
Mean pooling into concept vectors, compression ratio ablation, information retention.

**Papers:** DLCM ([2512.24617](https://arxiv.org/abs/2512.24617)) Eq. 7-10, Section 8 Table 5

In [ ]:
import torch, sys
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
sys.path.insert(0, '..')
from src.model import ConceptLMConfig, ConceptPooler, GlobalLoadBalancer, BoundaryDetector

## 1. Mean pooling: tokens → concept vectors (DLCM Eq. 7)

In [ ]:
cfg    = ConceptLMConfig(d_token=256, d_concept=512, d_scan=64)
pooler = ConceptPooler(cfg)

torch.manual_seed(0)
B, L = 1, 16
H = torch.randn(B, L, cfg.d_token)

b = torch.zeros(B, L)
for pos in [0, 4, 9, 13]:
    b[0, pos] = 1

C_list, seg_maps = pooler(H, b)
C = C_list[0]

print(f"Input : L={L} tokens, d_token={cfg.d_token}")
print(f"Output: M={C.shape[0]} concepts, d_concept={cfg.d_concept}")
print(f"Compression ratio: {L}/{C.shape[0]} = {L/C.shape[0]:.1f}x")
print(f"Segment map: {seg_maps[0].tolist()}")

## 2. Compression ratio ablation

In [ ]:
det = BoundaryDetector(cfg)
torch.manual_seed(42)
H_test = torch.randn(8, 64, cfg.d_token)

print(f"{'R (target)':12s} {'actual ratio':14s} {'aux_loss':10s}")
for R in [2, 4, 8, 16]:
    bal  = GlobalLoadBalancer(R)
    b2, p2 = det(H_test, training=True)
    loss = bal(b2, p2)
    n_b  = b2.sum(dim=1).float().mean()
    actual_R = 64 / n_b.item()
    print(f"  R={R:2d}           {actual_R:8.2f}       {loss.item():.5f}")

## 3. Adaptive granularity by content type (replicates DLCM Table 5)

In [ ]:
content_types = {
    "casual_english":   0.08,
    "technical_prose":  0.15,
    "code":             0.20,
    "math_science":     0.18,
}
L_sim, R_target = 100, 4
print(f"Target R={R_target} | Sequence length L={L_sim}")
print()
for name, p_boundary in content_types.items():
    n_b = max(1, int(L_sim * p_boundary))
    actual_ratio = L_sim / n_b
    print(f"  {name:22s}: p_boundary={p_boundary:.2f}  tokens/concept={actual_ratio:.1f}")
print()
print("Load balancer penalizes deviations from target R globally,")
print("while allowing per-sequence adaptive granularity.")

## 4. Information retention through pooling

In [ ]:
torch.manual_seed(1)
H2 = torch.randn(1, 20, cfg.d_token)
b2 = torch.zeros(1, 20); b2[0, 0] = 1; b2[0, 10] = 1
C2_list, _ = pooler(H2, b2)
C2 = C2_list[0]

c0_direct = pooler.W_up(H2[0, :10].mean(0))
c1_direct = pooler.W_up(H2[0, 10:].mean(0))

sim0 = F.cosine_similarity(C2[0].unsqueeze(0), c0_direct.unsqueeze(0)).item()
sim1 = F.cosine_similarity(C2[1].unsqueeze(0), c1_direct.unsqueeze(0)).item()
print(f"Concept 0 vs direct mean-pool projection: cosine sim = {sim0:.6f}")
print(f"Concept 1 vs direct mean-pool projection: cosine sim = {sim1:.6f}")
print("=> Concept vectors are exact projections of segment mean-pools.")